# SceneHawk - fill the feelings table

Runs the trained feelings models over every review in Supabase, averages each
review's scores across them, averages those per film, and writes
`film_feelings`. This is step 12 of the pipeline (`scripts/12_score_reviews.py`);
the notebook only sets it up on a GPU.

**Before you start**

1. Runtime > Change runtime type > **T4 GPU**.
2. The models from `train_feelings.ipynb`. `MODEL_NAMES` in the next cell says
   which ones: the three roberta-base f21 seeds picked on 2026-11-21 (same
   setup, seeds 49900/49901/49902), averaged with equal weights. One name in
   the list fills the table from that model alone.
   - `USE_DRIVE = False` (the default): no Drive access. In Google Drive,
     right-click each folder under `scenehawk/models/` > Download, which gives
     one zip per model (about 500 MB each; the zip's name can be anything). The
     next cell asks you to upload all three. Scores are cached in this session
     only, so a disconnect means scoring again from the start.
   - `USE_DRIVE = True`: the notebook mounts your Drive and reads the models
     from `MyDrive/scenehawk/models/`. Faster, and the cache is kept in Drive,
     so a disconnect resumes where it stopped.
3. Add your Supabase connection string as a Colab secret: the key icon in the
   left bar > **Add new secret**, name `SUPABASE_DB_URL`, value from Supabase >
   Connect > Connection pooling (with your password in it). Turn on notebook
   access. Never paste it into a cell.
4. `db/schema.sql` and `db/migrations/2026-11-06_search.sql` have been run in
   the Supabase SQL editor.

The code is cloned from `REPO` on GitHub, so changes to
`scripts/12_score_reviews.py` only take effect once they are pushed to
`BRANCH`. The averaging needs the 11/21 version of step 12 (it accepts several
`--model` folders). The scoring step prints how reviews are cut: for these
models it must say `head_tail`, and `21 feelings`.

**Two passes.** `DRY_RUN = True` (the default) scores everything and saves the
table to Drive without touching Supabase: about 15 minutes per model, so about
45 minutes for three. Check the printout, then set `DRY_RUN = False` and run the
last cell again. Every review is cached by then, so the second pass only
averages and writes, in a minute or two. If Colab disconnects, run all again.
With `USE_DRIVE = True` finished reviews are cached in Drive and skipped;
without it, re-upload the zips and it scores from the start.

In [ ]:
# Where the code comes from. Change these if the team repo moves.
REPO = "https://github.com/sroger123/SceneHawk-CS499-Project.git"
BRANCH = "main"

# The models to fill the table with, set by name: folder names do not sort by
# date. Several names = their scores are averaged with equal weights.
MODEL_NAMES = [
    "roberta_e12_headtail_f21_s49900-20261121-0724",
    "roberta_e12_headtail_f21_s49901-20261121-1726",
    "roberta_e12_headtail_f21_s49902-20261121-1754",
]
# Stored in film_feelings.model_version, so the website shows which model
# produced its scores.
VERSION = "roberta_e12_headtail_f21_seedavg3-20261121"
# True: score and save files only. False: also replace the rows in Supabase.
DRY_RUN = True
# False: upload a zip of each model folder (no Drive access).
# True: mount Google Drive and read them from MyDrive/scenehawk/models/.
USE_DRIVE = False

import os, pathlib, shutil, zipfile
os.chdir("/content")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/scenehawk/models")
else:
    ROOT = pathlib.Path("/content/models")
    ROOT.mkdir(parents=True, exist_ok=True)
    missing = [n for n in MODEL_NAMES if not (ROOT / n / "config.json").exists()]
    if missing:
        from google.colab import files
        print("Upload one zip per model (any file names):", ", ".join(missing))
        uploaded = files.upload()          # saved into /content
        for zname in uploaded:
            tmp = ROOT / "_unzip"
            shutil.rmtree(tmp, ignore_errors=True)
            with zipfile.ZipFile(zname) as z:
                z.extractall(tmp)
            os.remove(zname)
            for cfg in sorted(tmp.rglob("config.json")):
                # Match by the folder name inside the zip, else by the zip's name.
                hits = [n for n in MODEL_NAMES
                        if n in cfg.parent.name or n in zname]
                assert len(hits) == 1, (f"{zname}: cannot tell which model "
                                        f"{cfg.parent.name} is; rename the zip "
                                        "to the model name")
                shutil.rmtree(ROOT / hits[0], ignore_errors=True)
                shutil.move(str(cfg.parent), str(ROOT / hits[0]))
                print("unpacked", hits[0])
            shutil.rmtree(tmp, ignore_errors=True)

print("models found:")
for m in sorted(ROOT.glob("*/config.json")):
    print(" ", m.parent.name)
for name in MODEL_NAMES:
    assert (ROOT / name / "config.json").exists(), (
        f"{name} not found in {ROOT}" + ("" if USE_DRIVE else ": upload its zip "
                                         "and run this cell again"))
MODEL_DIRS = " ".join(f'"{ROOT / name}"' for name in MODEL_NAMES)
print("using:", ", ".join(MODEL_NAMES))
print("version:", VERSION, "| dry run" if DRY_RUN else "| WRITES TO SUPABASE")

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} {REPO} /content/repo
!pip -q install psycopg2-binary python-dotenv

import os
from google.colab import userdata
os.environ["SUPABASE_DB_URL"] = userdata.get("SUPABASE_DB_URL")

# Per-review cache: in Drive a disconnect doesn't lose work; without Drive it
# lasts as long as this Colab session.
cache = ("/content/drive/MyDrive/scenehawk/feelings" if USE_DRIVE
         else "/content/feelings")
os.makedirs(cache, exist_ok=True)
!rm -rf /content/repo/chatbot/data/feelings && ln -s {cache} /content/repo/chatbot/data/feelings

## Score the reviews and write the table

Each model prints its name and the device (it should say `cuda`) before it
scores, then the scores are averaged. At the end it prints the top films for a
few feelings as a sanity check, and `dry run: nothing written to Supabase` while
`DRY_RUN` is on.

In [ ]:
%cd /content/repo/chatbot
DRY = "--dry-run" if DRY_RUN else ""
!python scripts/12_score_reviews.py --model {MODEL_DIRS} --version "{VERSION}" --batch 64 {DRY}

## Then

- The website's chat page now answers from the new scores. Check it at
  `<your site>/api/chat`: it reports how many films have feeling scores and
  the `model_version`, which should be `VERSION` above.
- Retrained the models? Update `MODEL_NAMES` and `VERSION` and run this
  notebook again. Each model writes its own cache file, and the table is
  replaced film by film.